# Notebook para clasificación multietiqueta de textos médicos usando los datos de Codiesp

Este notebook aborda el problema de clasificación multietiqueta de textos médicos, empleando los datos proporcionados por Codiesp. Se utilizarán técnicas de Deep Learning y procesamiento de lenguaje natural para construir un modelo capaz de asignar múltiples etiquetas CIE-10 a cada texto clínico. El flujo incluye análisis estadístico, preprocesamiento, tokenización, construcción y entrenamiento del modelo, así como la evaluación y conclusiones.

## 1. Carga y exploración de los datos

En esta sección se cargarán los archivos CSV de entrenamiento, validación y test. Se mostrarán ejemplos de textos y etiquetas, y se verificará la estructura de los datos.

In [ ]:
import pandas as pd

# Rutas de los archivos
train_path = "/home/david/workspace/CIE-10/train.csv"
val_path = "/home/david/workspace/CIE-10/validation.csv"
test_path = "/home/david/workspace/CIE-10/test.csv"

# Carga de los datos
df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)
df_test = pd.read_csv(test_path)

# Visualización de ejemplos
print("Ejemplo de fila en train:")
print(df_train.iloc[0])

print("\nColumnas disponibles:", df_train.columns.tolist())

## 2. Análisis estadístico de los ficheros

Se analizará la distribución de etiquetas, longitud de los textos, frecuencia de etiquetas por muestra, y se identificarán las etiquetas más y menos frecuentes. Se visualizarán estadísticas con gráficos.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import ast

# Función para convertir la columna de etiquetas de string a lista
def parse_labels(labels_str):
    try:
        return ast.literal_eval(labels_str)
    except:
        return []

df_train['Labels_list'] = df_train['Labels'].apply(parse_labels)
df_val['Labels_list'] = df_val['Labels'].apply(parse_labels)
df_test['Labels_list'] = df_test['Labels'].apply(parse_labels)

# Longitud de textos
df_train['text_length'] = df_train['Text'].apply(lambda x: len(str(x).split()))

plt.figure(figsize=(8,4))
sns.histplot(df_train['text_length'], bins=30, kde=True)
plt.title("Distribución de la longitud de los textos (train)")
plt.xlabel("Número de palabras")
plt.ylabel("Frecuencia")
plt.show()

# Número de etiquetas por muestra
df_train['num_labels'] = df_train['Labels_list'].apply(len)
plt.figure(figsize=(8,4))
sns.histplot(df_train['num_labels'], bins=20, kde=True)
plt.title("Distribución del número de etiquetas por muestra (train)")
plt.xlabel("Número de etiquetas")
plt.ylabel("Frecuencia")
plt.show()

# Frecuencia de etiquetas
from collections import Counter

all_labels = [label for labels in df_train['Labels_list'] for label in labels]
label_counts = Counter(all_labels)
labels, counts = zip(*label_counts.most_common())

plt.figure(figsize=(12,5))
sns.barplot(x=list(labels)[:20], y=list(counts)[:20])
plt.title("Top 20 etiquetas más frecuentes")
plt.xlabel("Etiqueta")
plt.ylabel("Frecuencia")
plt.xticks(rotation=45)
plt.show()

print("Número total de etiquetas distintas en train:", len(label_counts))
print("Etiquetas menos frecuentes:", [l for l,c in label_counts.items() if c==1])

## 3. Preprocesamiento de textos y etiquetas

Se limpiarán los textos eliminando caracteres especiales y normalizando el contenido. Las etiquetas se procesarán para el formato multietiqueta.

In [ ]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"[^a-záéíóúüñ0-9.,;:/\- ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

df_train['Text_clean'] = df_train['Text'].apply(clean_text)
df_val['Text_clean'] = df_val['Text'].apply(clean_text)
df_test['Text_clean'] = df_test['Text'].apply(clean_text)

print("Texto original:")
print(df_train['Text'].iloc[0])
print("\nTexto limpio:")
print(df_train['Text_clean'].iloc[0])

## 4. Tokenización y vectorización de textos

Se utilizará la librería HuggingFace Transformers para tokenizar los textos médicos y convertirlos en tensores para el modelo.

In [ ]:
from transformers import AutoTokenizer

# Usar un modelo preentrenado relevante (ejemplo: bert-base-multilingual-cased o ClinicalBERT)
MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Ejemplo de tokenización
encoded = tokenizer(df_train['Text_clean'].iloc[0], truncation=True, padding='max_length', max_length=256)
print("Tokens:", encoded['input_ids'][:20])
print("Tokenized text:", tokenizer.convert_ids_to_tokens(encoded['input_ids'][:20]))

## 5. Preparación de las etiquetas multietiqueta

Las listas de etiquetas se convertirán en vectores binarios para clasificación multietiqueta usando MultiLabelBinarizer de scikit-learn.

In [ ]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit(df_train['Labels_list'])

y_train = mlb.transform(df_train['Labels_list'])
y_val = mlb.transform(df_val['Labels_list'])
y_test = mlb.transform(df_test['Labels_list'])

print("Ejemplo de vector binario de etiquetas:", y_train[0])
print("Número de etiquetas posibles:", len(mlb.classes_))

## 6. División de los datos en conjuntos de entrenamiento, validación y test

Se asegura que los datos estén correctamente divididos y listos para el entrenamiento y evaluación.

In [ ]:
# Preparación de los textos tokenizados para el modelo
def tokenize_texts(texts, tokenizer, max_length=256):
    return tokenizer(
        list(texts),
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

X_train = tokenize_texts(df_train['Text_clean'], tokenizer)
X_val = tokenize_texts(df_val['Text_clean'], tokenizer)
X_test = tokenize_texts(df_test['Text_clean'], tokenizer)

print("Tamaño de X_train:", X_train['input_ids'].shape)
print("Tamaño de y_train:", y_train.shape)

## 7. Construcción del modelo de clasificación multietiqueta (Deep Learning)

Se define un modelo basado en Transformers (por ejemplo, BERT) con una capa de salida para clasificación multietiqueta. Se explica la arquitectura y la función de pérdida (Binary Crossentropy).

In [ ]:
import torch
from torch import nn
from transformers import AutoModel

class MultiLabelClassifier(nn.Module):
    def __init__(self, model_name, num_labels):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.classifier(x)
        return logits

num_labels = len(mlb.classes_)
model = MultiLabelClassifier(MODEL_NAME, num_labels)

# Explicación de la función de pérdida
loss_fn = nn.BCEWithLogitsLoss()

## 8. Entrenamiento del modelo en GPU (RTX 4080)

Se configura el entrenamiento para usar la GPU. Se entrena el modelo y se muestra el progreso (loss, accuracy, etc.).

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MedicalTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = MedicalTextDataset(X_train, y_train)
val_dataset = MedicalTextDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

from tqdm import tqdm

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def train_epoch(model, loader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for batch in tqdm(loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

# Entrenamiento de ejemplo (1 epoch)
epoch_loss = train_epoch(model, train_loader, optimizer, loss_fn, device)
print("Loss de entrenamiento (1 epoch):", epoch_loss)

## 9. Evaluación del modelo y métricas multietiqueta

Se evalúa el modelo en el conjunto de test usando métricas como F1-score, precisión, recall y hamming loss. Se muestran ejemplos de predicciones.

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score, hamming_loss

model.eval()
test_dataset = MedicalTextDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=8)

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()
        logits = model(input_ids, attention_mask)
        preds = torch.sigmoid(logits).cpu().numpy()
        preds_bin = (preds > 0.5).astype(int)
        all_preds.append(preds_bin)
        all_labels.append(labels)

import numpy as np
all_preds = np.vstack(all_preds)
all_labels = np.vstack(all_labels)

f1 = f1_score(all_labels, all_preds, average='micro')
precision = precision_score(all_labels, all_preds, average='micro')
recall = recall_score(all_labels, all_preds, average='micro')
hamming = hamming_loss(all_labels, all_preds)

print(f"F1-score (micro): {f1:.4f}")
print(f"Precisión (micro): {precision:.4f}")
print(f"Recall (micro): {recall:.4f}")
print(f"Hamming loss: {hamming:.4f}")

# Ejemplo de predicción
idx = 0
pred_labels = [mlb.classes_[i] for i, v in enumerate(all_preds[idx]) if v == 1]
true_labels = [mlb.classes_[i] for i, v in enumerate(all_labels[idx]) if v == 1]
print("Texto de test:", df_test['Text'].iloc[idx])
print("Etiquetas reales:", true_labels)
print("Etiquetas predichas:", pred_labels)

## 10. Análisis de resultados y conclusiones

En esta sección se analizan los resultados obtenidos, se identifican posibles mejoras y se limita el alcance del modelo. Se extraen conclusiones sobre el rendimiento y utilidad del clasificador multietiqueta aplicado a textos médicos.

- El modelo logra identificar múltiples etiquetas por texto, aunque la precisión depende de la frecuencia y representación de cada etiqueta en los datos.
- Las etiquetas menos frecuentes presentan mayor dificultad de predicción, lo que sugiere la necesidad de técnicas de balanceo o aumento de datos.
- El preprocesamiento y la tokenización adecuada son claves para el rendimiento en textos médicos, que suelen ser complejos y heterogéneos.
- El uso de modelos preentrenados como BERT permite aprovechar el contexto lingüístico, pero podría mejorarse usando modelos específicos del dominio clínico (ej. ClinicalBERT).
- El entrenamiento en GPU (RTX 4080) permite acelerar el proceso y manejar grandes volúmenes de datos.
- Para futuras mejoras, se recomienda explorar técnicas de data augmentation, ajuste de hiperparámetros y modelos más avanzados.

Este clasificador puede ser útil como herramienta de apoyo en la codificación automática de textos médicos, facilitando la asignación de códigos CIE-10 en entornos clínicos.

## 11. Enfoque jerárquico: clasificación por niveles de los códigos CIE-10

Los códigos CIE-10 tienen una estructura jerárquica (letra, números, sufijos). Una estrategia posible es entrenar un modelo independiente para cada nivel. Por ejemplo, primero clasificar la letra (familia principal), luego el número (subfamilia) y finalmente el sufijo (detalle).

### ¿Entrenar un modelo para cada nivel?

**Ventajas:**
- Permite especializar cada modelo en un nivel concreto, facilitando el aprendizaje de patrones específicos.
- Puede mejorar la precisión en niveles superiores (letra), que suelen tener menos clases y más ejemplos por clase.
- Facilita la interpretación y el análisis de errores por nivel.

**Desventajas:**
- La semántica de los números y sufijos depende de la letra: el mismo número en diferentes familias significa cosas distintas.
- Requiere coordinar las predicciones entre modelos, lo que puede aumentar la complejidad.
- Si el modelo de la letra falla, los modelos siguientes pueden recibir información incorrecta.

### Ejemplo: Clasificación de la letra (familia principal)

A continuación se muestra cómo extraer la letra de cada código y entrenar un clasificador para este primer nivel.

In [ ]:
# Extraer la letra principal de cada código

def extract_family(codes):
    return list(set([str(code)[0] for code in codes]))

# Suponiendo que tienes los DataFrames df_train, df_val, df_test con la columna 'Labels_list'
df_train['Label_family'] = df_train['Labels_list'].apply(extract_family)
df_val['Label_family'] = df_val['Labels_list'].apply(extract_family)
df_test['Label_family'] = df_test['Labels_list'].apply(extract_family)

# Binarización multietiqueta para la letra
from sklearn.preprocessing import MultiLabelBinarizer
mlb_family = MultiLabelBinarizer()
mlb_family.fit(df_train['Label_family'])

y_train_family = mlb_family.transform(df_train['Label_family'])
y_val_family = mlb_family.transform(df_val['Label_family'])
y_test_family = mlb_family.transform(df_test['Label_family'])

print("Familias posibles:", mlb_family.classes_)
print("Ejemplo de vector binario de familias:", y_train_family[0])

In [ ]:
# Entrenamiento del modelo para la letra
# Reutiliza la arquitectura del modelo multietiqueta, cambiando el número de salidas por el número de familias posibles

num_families = len(mlb_family.classes_)
model_family = MultiLabelClassifier(MODEL_NAME, num_families)
model_family.to(device)

# Dataset y DataLoader para la letra
train_dataset_family = MedicalTextDataset(X_train, y_train_family)
val_dataset_family = MedicalTextDataset(X_val, y_val_family)
train_loader_family = DataLoader(train_dataset_family, batch_size=8, shuffle=True)
val_loader_family = DataLoader(val_dataset_family, batch_size=8)

# Entrenamiento de ejemplo (1 epoch)
optimizer_family = torch.optim.AdamW(model_family.parameters(), lr=2e-5)
epoch_loss_family = train_epoch(model_family, train_loader_family, optimizer_family, loss_fn, device)
print("Loss de entrenamiento (familia, 1 epoch):", epoch_loss_family)

In [ ]:
# Evaluación del modelo de la letra
from sklearn.metrics import f1_score
import numpy as np
from tqdm import tqdm

# Evaluación en test
test_dataset_family = MedicalTextDataset(X_test, y_test_family)
test_loader_family = DataLoader(test_dataset_family, batch_size=8)

all_preds_family = []
all_labels_family = []

model_family.eval()
with torch.no_grad():
    for batch in tqdm(test_loader_family):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].cpu().numpy()
        logits = model_family(input_ids, attention_mask)
        preds = torch.sigmoid(logits).cpu().numpy()
        preds_bin = (preds > 0.5).astype(int)
        all_preds_family.append(preds_bin)
        all_labels_family.append(labels)

all_preds_family = np.vstack(all_preds_family)
all_labels_family = np.vstack(all_labels_family)

f1_family = f1_score(all_labels_family, all_preds_family, average='micro')
print(f"F1-score (familia, micro): {f1_family:.4f}")

---

## 12. Ventajas y desventajas del enfoque jerárquico

**Ventajas:**
- Aprovecha la estructura semántica de los códigos.
- Puede facilitar el aprendizaje y la interpretación.

**Desventajas:**
- La semántica de los subniveles depende de la familia principal.
- Aumenta la complejidad y el riesgo de errores encadenados.

**Conclusión:**  
El enfoque jerárquico puede ser útil para analizar y mejorar el rendimiento en cada nivel, pero requiere un diseño cuidadoso para evitar errores de propagación y problemas semánticos entre niveles. Se recomienda comparar este enfoque con el modelo plano y analizar los resultados.

## 13. Clasificación jerárquica: modelo secuencial por niveles

Para aprovechar la jerarquía de los códigos CIE-10, se propone entrenar un modelo para cada nivel:
- **Nivel 1:** Predicción de la letra (familia principal)
- **Nivel 2:** Predicción del número (subfamilia), condicionado a la familia
- **Nivel 3:** Predicción del sufijo/detalle, condicionado a familia y subfamilia

Este enfoque permite que cada modelo aprenda patrones específicos de su nivel y aproveche la información de los niveles anteriores.

In [ ]:
# Descomposición jerárquica de los códigos
import re

def split_code(code):
    match = re.match(r"([A-Z])([0-9]+)\.?([a-z0-9]*)", str(code))
    if match:
        family, subfamily, detail = match.groups()
        return family, subfamily, detail
    else:
        return code, "", ""

# Aplicar descomposición a todas las etiquetas
def decompose_labels(label_list):
    families, subfamilies, details = [], [], []
    for code in label_list:
        f, s, d = split_code(code)
        families.append(f)
        subfamilies.append(f + s)
        details.append(f + s + d)
    return list(set(families)), list(set(subfamilies)), list(set(details))

# Añadir columnas jerárquicas
df_train[['Label_family', 'Label_subfamily', 'Label_detail']] = df_train['Labels_list'].apply(lambda x: pd.Series(decompose_labels(x)))
df_val[['Label_family', 'Label_subfamily', 'Label_detail']] = df_val['Labels_list'].apply(lambda x: pd.Series(decompose_labels(x)))
df_test[['Label_family', 'Label_subfamily', 'Label_detail']] = df_test['Labels_list'].apply(lambda x: pd.Series(decompose_labels(x)))

# Binarización multietiqueta para cada nivel
mlb_family = MultiLabelBinarizer().fit(df_train['Label_family'])
mlb_subfamily = MultiLabelBinarizer().fit(df_train['Label_subfamily'])
mlb_detail = MultiLabelBinarizer().fit(df_train['Label_detail'])

y_train_family = mlb_family.transform(df_train['Label_family'])
y_train_subfamily = mlb_subfamily.transform(df_train['Label_subfamily'])
y_train_detail = mlb_detail.transform(df_train['Label_detail'])

y_val_family = mlb_family.transform(df_val['Label_family'])
y_val_subfamily = mlb_subfamily.transform(df_val['Label_subfamily'])
y_val_detail = mlb_detail.transform(df_val['Label_detail'])

y_test_family = mlb_family.transform(df_test['Label_family'])
y_test_subfamily = mlb_subfamily.transform(df_test['Label_subfamily'])
y_test_detail = mlb_detail.transform(df_test['Label_detail'])

print("Ejemplo de familias:", df_train['Label_family'].iloc[0])
print("Ejemplo de subfamilias:", df_train['Label_subfamily'].iloc[0])
print("Ejemplo de detalles:", df_train['Label_detail'].iloc[0])

In [ ]:
# Entrenamiento de modelos jerárquicos
# Modelo para familia
num_families = len(mlb_family.classes_)
model_family = MultiLabelClassifier(MODEL_NAME, num_families)
model_family.to(device)
train_dataset_family = MedicalTextDataset(X_train, y_train_family)
train_loader_family = DataLoader(train_dataset_family, batch_size=8, shuffle=True)
optimizer_family = torch.optim.AdamW(model_family.parameters(), lr=2e-5)
train_epoch(model_family, train_loader_family, optimizer_family, loss_fn, device)

# Modelo para subfamilia
num_subfamilies = len(mlb_subfamily.classes_)
model_subfamily = MultiLabelClassifier(MODEL_NAME, num_subfamilies)
model_subfamily.to(device)
train_dataset_subfamily = MedicalTextDataset(X_train, y_train_subfamily)
train_loader_subfamily = DataLoader(train_dataset_subfamily, batch_size=8, shuffle=True)
optimizer_subfamily = torch.optim.AdamW(model_subfamily.parameters(), lr=2e-5)
train_epoch(model_subfamily, train_loader_subfamily, optimizer_subfamily, loss_fn, device)

# Modelo para detalle
num_details = len(mlb_detail.classes_)
model_detail = MultiLabelClassifier(MODEL_NAME, num_details)
model_detail.to(device)
train_dataset_detail = MedicalTextDataset(X_train, y_train_detail)
train_loader_detail = DataLoader(train_dataset_detail, batch_size=8, shuffle=True)
optimizer_detail = torch.optim.AdamW(model_detail.parameters(), lr=2e-5)
train_epoch(model_detail, train_loader_detail, optimizer_detail, loss_fn, device)

print("Modelos jerárquicos entrenados para cada nivel.")

### Ventajas del modelo jerárquico secuencial
- Aprovecha la estructura semántica de los códigos.
- Permite especializar cada modelo en su nivel.
- Facilita el análisis de errores por nivel.

### Desventajas
- Si un modelo falla en un nivel, los siguientes pueden recibir información incorrecta.
- Aumenta la complejidad y el coste computacional.
- Requiere coordinación entre modelos y manejo especial de dependencias semánticas.

**Conclusión:** El modelo jerárquico secuencial puede mejorar la interpretación y el rendimiento en problemas con estructura jerárquica, pero requiere un diseño y análisis cuidadoso.

## 14. Evaluación y comparación: modelo plano vs modelo jerárquico

A continuación se evalúan ambos enfoques y se comparan sus resultados en los distintos niveles de la jerarquía.

In [ ]:
# Evaluación del modelo plano (multilabel estándar)
from sklearn.metrics import f1_score, classification_report

# Suponiendo que tienes all_preds_flat y all_labels_flat para el modelo plano
f1_flat = f1_score(all_labels_flat, all_preds_flat, average='micro')
print(f"F1-score modelo plano (micro): {f1_flat:.4f}")
print(classification_report(all_labels_flat, all_preds_flat))

# Evaluación de los modelos jerárquicos
# Familia
f1_family = f1_score(all_labels_family, all_preds_family, average='micro')
print(f"F1-score modelo jerárquico (familia, micro): {f1_family:.4f}")
# Subfamilia
f1_subfamily = f1_score(all_labels_subfamily, all_preds_subfamily, average='micro')
print(f"F1-score modelo jerárquico (subfamilia, micro): {f1_subfamily:.4f}")
# Detalle
f1_detail = f1_score(all_labels_detail, all_preds_detail, average='micro')
print(f"F1-score modelo jerárquico (detalle, micro): {f1_detail:.4f}")

### Interpretación de resultados

- El modelo plano trata todas las etiquetas como independientes, lo que puede dificultar la predicción de etiquetas raras o dependientes.
- El modelo jerárquico puede mejorar la precisión en los niveles superiores y facilitar la interpretación de errores.
- Es recomendable analizar los resultados por nivel y por tipo de error para decidir el enfoque más adecuado según el problema y los datos.

In [ ]:
# Visualización de resultados
import matplotlib.pyplot as plt
import seaborn as sns

# F1-scores de cada modelo
scores = {
    'Plano': f1_flat,
    'Jerárquico - Familia': f1_family,
    'Jerárquico - Subfamilia': f1_subfamily,
    'Jerárquico - Detalle': f1_detail
}

plt.figure(figsize=(8,5))
sns.barplot(x=list(scores.keys()), y=list(scores.values()), palette='viridis')
plt.title('Comparación de F1-score entre modelos')
plt.ylabel('F1-score (micro)')
plt.ylim(0,1)
plt.show()

# Matriz de confusión para el modelo plano (opcional, solo para pocas clases)
#from sklearn.metrics import multilabel_confusion_matrix
#cm = multilabel_confusion_matrix(all_labels_flat, all_preds_flat)
#plt.figure(figsize=(12,8))
#for i, label in enumerate(mlb_flat.classes_):
#    plt.subplot(3,6,i+1)
#    sns.heatmap(cm[i], annot=True, fmt='d', cbar=False)
#    plt.title(label)
#plt.tight_layout()
#plt.show()